In [2]:
# 1. Importok és modellek betöltése
import pandas as pd
import numpy as np
import joblib
import pickle
from sklearn.preprocessing import RobustScaler
import matplotlib.pyplot as plt
import os

# Betöltés a legújabb modellfájlokból
import glob
def load_latest_model(pattern):
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"No files found for pattern: {pattern}")
    latest_file = max(files, key=os.path.getctime)
    return joblib.load(latest_file)

# Modellek betöltése
try:
    # Betöltjük a fő artifact file-t
    model_files = glob.glob('models/football_model_*_artifacts.pkl')
    if model_files:
        latest_model = max(model_files, key=os.path.getctime)
        with open(latest_model, 'rb') as f:
            model_artifacts = pickle.load(f)
        
        models = model_artifacts['models']
        scaler = model_artifacts['scaler']
        feature_columns = model_artifacts['feature_columns']
        
        print("Modellek sikeresen betöltve!")
        print(f"Elérhető modellek: {list(models.keys())}")
        print(f"Feature columns: {feature_columns}")
    else:
        print("Nincsenek modell fájlok!")
        
except Exception as e:
    print(f"Hiba a modellek betöltésekor: {e}")

Modellek sikeresen betöltve!
Elérhető modellek: ['RandomForest', 'GradientBoosting', 'LogisticRegression']
Feature columns: ['Home_Implied_Prob', 'Draw_Implied_Prob', 'Away_Implied_Prob', 'Home_Points_Last_5', 'Home_Goals_For_Last_5', 'Home_Goals_Against_Last_5', 'Away_Points_Last_5', 'Away_Goals_For_Last_5', 'Away_Goals_Against_Last_5', 'Home_Form_Last_5', 'Away_Form_Last_5', 'Home_Advantage', 'Days_Since_Last_Home_Match', 'Days_Since_Last_Away_Match']


In [3]:
# 2. Új adatok betöltése és előkészítése
def prepare_new_data(match_data, feature_columns):
    """
    Előkészíti az új mérkőzés adatokat predikcióra
    """
    # Alap feature engineering (ugyanaz, mint a trainingnél)
    df = pd.DataFrame([match_data])
    
    # Data cleaning
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
    
    # Implied probabilities
    if 'AvgH' in df.columns and 'AvgD' in df.columns and 'AvgA' in df.columns:
        df['Home_Implied_Prob'] = 1 / np.clip(df['AvgH'], 1.01, 1000)
        df['Draw_Implied_Prob'] = 1 / np.clip(df['AvgD'], 1.01, 1000)
        df['Away_Implied_Prob'] = 1 / np.clip(df['AvgA'], 1.01, 1000)
    
    # Hiányzó értékek kezelése
    for col in feature_columns:
        if col not in df.columns:
            df[col] = 0  # Default érték
    
    # Csak a szükséges oszlopok
    X_new = df[feature_columns]
    
    # Skálázás
    X_scaled = scaler.transform(X_new)
    
    return X_scaled, df

# Példa új mérkőzés adatokra
example_match = {
    'Date': '2024-01-15',
    'HomeTeam': 'Manchester United',
    'AwayTeam': 'Liverpool',
    'AvgH': 2.5,
    'AvgD': 3.2,
    'AvgA': 2.8,
    'Home_Points_Last_5': 2.1,
    'Away_Points_Last_5': 2.8,
    'Home_Goals_For_Last_5': 1.4,
    'Away_Goals_For_Last_5': 2.1,
    'Home_Goals_Against_Last_5': 1.2,
    'Away_Goals_Against_Last_5': 0.8,
    'Home_Form_Last_5': 8,
    'Away_Form_Last_5': 12,
    'Home_Advantage': 1,
    'Days_Since_Last_Home_Match': 7,
    'Days_Since_Last_Away_Match': 6
}

X_new, prepared_df = prepare_new_data(example_match, feature_columns)
print("Adatok előkészítve predikcióra")

Adatok előkészítve predikcióra


C:\Users\Adam\AppData\Local\Temp\ipykernel_20764\470502439.py:10: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')


In [9]:
# 3. Predikció készítése
def predict_match(models, X_new, odds_data=None, threshold=0.05):
    """
    Predikció készítése az összes modelllel
    """
    predictions = {}
    
    for model_name, model in models.items():
        try:
            # Valószínűségek predikálása
            probs = model.predict_proba(X_new)[0]  # Csak az első mérkőzés
            pred_class = model.predict(X_new)[0]
            
            # Value bet számítás
            value_bets = {}
            if odds_data is not None:
                home_implied = 1 / odds_data['AvgH']
                draw_implied = 1 / odds_data['AvgD'] 
                away_implied = 1 / odds_data['AvgA']
                
                value_bets = {
                    'Home_Value': probs[0] > home_implied + threshold,
                    'Draw_Value': probs[1] > draw_implied + threshold,
                    'Away_Value': probs[2] > away_implied + threshold
                }
            
            predictions[model_name] = {
                'probabilities': probs,
                'predicted_class': pred_class,
                'value_bets': value_bets,
                'recommendation': get_recommendation(probs, value_bets, odds_data)
            }
            
        except Exception as e:
            print(f"Hiba {model_name} predikciójánál: {e}")
            predictions[model_name] = None
    
    return predictions

def get_recommendation(probs, value_bets, odds_data):
    """
    Ajánlás generálása a predikció alapján
    """
    if not value_bets:
        return "Nincs elég adat az ajánláshoz"
    
    recommendations = []
    outcomes = ['Home', 'Draw', 'Away']
    
    for i, outcome in enumerate(outcomes):
        if value_bets[f'{outcome}_Value']:
            stake = calculate_stake(probs[i], odds_data[f'Avg{outcome[0]}'])
            recommendations.append(f"{outcome} win: {probs[i]:.1%} valószínűség")
    
    return " | ".join(recommendations) if recommendations else "Nincs érték fogadás"

def calculate_stake(prob, odds, bankroll=1000, risk_percent=0.02):
    """
    Tét méretének számítása
    """
    kelly_fraction = (prob * odds - 1) / (odds - 1) if odds > 1 else 0
    return max(0, min(kelly_fraction, 0.1)) * bankroll * risk_percent

# Predikció készítése
odds_data = {
    'AvgH': example_match['AvgH'],
    'AvgD': example_match['AvgD'], 
    'AvgA': example_match['AvgA']
}

predictions = predict_match(models, X_new, odds_data)
print("Predikció kész!")

Predikció kész!


c:\Users\Adam\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Adam\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Adam\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
c:\Users\Adam\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
c:\Users\Adam\AppData\Local\Programs\Python\Python313\Lib\site-packages\

In [10]:
# 4. Eredmények megjelenítése
def display_predictions(predictions, odds_data):
    """
    Predikciók szép megjelenítése
    """
    print("🎯 PREDIKCIÓS EREDMÉNYEK")
    print("=" * 50)
    
    for model_name, pred in predictions.items():
        if pred is None:
            continue
            
        print(f"\n📊 {model_name}:")
        print(f"   🏠 Home: {pred['probabilities'][0]:.1%} (fair odds: {1/pred['probabilities'][0]:.2})")
        print(f"   🤝 Draw: {pred['probabilities'][1]:.1%} (fair odds: {1/pred['probabilities'][1]:.2})") 
        print(f"   🚗 Away: {pred['probabilities'][2]:.1%} (fair odds: {1/pred['probabilities'][2]:.2})")
        
        if pred['value_bets']:
            print("   💰 Value Bets:")
            for outcome, is_value in pred['value_bets'].items():
                if is_value:
                    print(f"     ✅ {outcome}")
        
        print(f"   💡 Ajánlás: {pred['recommendation']}")
    
    print(f"\n📈 Odds: Home {odds_data['AvgH']:.2f} | Draw {odds_data['AvgD']:.2f} | Away {odds_data['AvgA']:.2f}")

# Eredmények megjelenítése
display_predictions(predictions, odds_data)

🎯 PREDIKCIÓS EREDMÉNYEK

📊 RandomForest:
   🏠 Home: 20.9% (fair odds: 4.8)
   🤝 Draw: 21.6% (fair odds: 4.6)
   🚗 Away: 57.5% (fair odds: 1.7)
   💰 Value Bets:
     ✅ Away_Value
   💡 Ajánlás: Away win: 57.5% valószínűség

📊 GradientBoosting:
   🏠 Home: 10.0% (fair odds: 1e+01)
   🤝 Draw: 12.7% (fair odds: 7.9)
   🚗 Away: 77.3% (fair odds: 1.3)
   💰 Value Bets:
     ✅ Away_Value
   💡 Ajánlás: Away win: 77.3% valószínűség

📊 LogisticRegression:
   🏠 Home: 16.6% (fair odds: 6.0)
   🤝 Draw: 10.6% (fair odds: 9.5)
   🚗 Away: 72.8% (fair odds: 1.4)
   💰 Value Bets:
     ✅ Away_Value
   💡 Ajánlás: Away win: 72.8% valószínűség

📈 Odds: Home 2.50 | Draw 3.20 | Away 2.80
